In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    _p = _root / "paths.py"
    if _p.exists() and "GPT4O_ROOT" in _p.read_text(encoding="utf-8", errors="ignore"):
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find GPT4o/paths.py — set Jupyter cwd to After_PT_Removal/GPT4o or a subfolder."
    )
import paths


In [1]:
import openai

import pandas as pd
import os
import re
import numpy as np
from difflib import SequenceMatcher
from tqdm import tqdm

output_file = str(paths.PREDICTIONS / "gpt4o_predictions_on_llama70b_removed.csv")

# Analysis of the predictions
import pandas as pd
import numpy as np
import re

# Load the predictions file
results_df = pd.read_csv(output_file)
print(f"Loaded {len(results_df)} rows from {output_file}")

# Function to extract letter from GPT4o prediction
def extract_answer_letter(prediction):
    if pd.isna(prediction) or prediction == "Error" or prediction == "Missing data":
        return None
    
    # Try to extract from <answer>Option [letter]</answer> format
    match = re.search(r'<answer>Option ([A-E])</answer>', prediction)
    if match:
        return match.group(1)
    
    # Alternative formats
    match = re.search(r'Option ([A-E])', prediction)
    if match:
        return match.group(1)
    
    # Just find any letter
    match = re.search(r'\b([A-E])\b', prediction)
    if match:
        return match.group(1)
    
    return None

# Extract answer letters
results_df['extracted_prediction'] = results_df['GPT4o_prediction'].apply(extract_answer_letter)

# Calculate accuracy
results_df['is_correct'] = results_df['extracted_prediction'] == results_df['answer_df3']

# Overall accuracy
valid_predictions = results_df['extracted_prediction'].notna()
overall_accuracy = results_df.loc[valid_predictions, 'is_correct'].mean() * 100
overall_std = results_df.loc[valid_predictions, 'is_correct'].std() * 100

print(f"\nOverall Accuracy: {overall_accuracy:.2f}%")
print(f"Standard Deviation: {overall_std:.2f}%")
print(f"Total valid predictions: {valid_predictions.sum()} ({valid_predictions.sum()/len(results_df)*100:.2f}%)")

# Analysis by data source
print("\nAccuracy by Data Source:")
data_sources = results_df['data_source_corr'].unique()

for source in data_sources:
    source_df = results_df[results_df['data_source_corr'] == source]
    valid_source_predictions = source_df['extracted_prediction'].notna()
    
    if valid_source_predictions.sum() > 0:
        source_accuracy = source_df.loc[valid_source_predictions, 'is_correct'].mean() * 100
        source_std = source_df.loc[valid_source_predictions, 'is_correct'].std() * 100
        source_count = valid_source_predictions.sum()
        
        print(f"{source}: {source_accuracy:.2f}% (±{source_std:.2f}%), n={source_count}")

# Save the analysis results
results_df.to_csv(paths.REMOVE_LOW_IRR_DATA / "GPT4o_analysis_results.csv", index=False)
print("\nDetailed analysis saved to GPT4o_analysis_results.csv")

Loaded 1297 rows from gpt4o_predictions_on_llama70b_removed.csv

Overall Accuracy: 69.79%
Standard Deviation: 45.94%
Total valid predictions: 1142 (88.05%)

Accuracy by Data Source:
jama: 69.24% (±46.19%), n=582
medxpert: 38.79% (±48.88%), n=165
medbullets: 74.63% (±43.62%), n=205
mmlu: 93.16% (±25.31%), n=190

Detailed analysis saved to GPT4o_analysis_results.csv


In [2]:
output_file = str(paths.REMOVE_LOW_IRR_DATA / "GPT4o_predictions.csv")

# Analysis of the predictions
import pandas as pd
import numpy as np
import re

# Load the predictions file
results_df = pd.read_csv(output_file)
print(f"Loaded {len(results_df)} rows from {output_file}")

# Function to extract letter from GPT4o prediction
def extract_answer_letter(prediction):
    if pd.isna(prediction) or prediction == "Error" or prediction == "Missing data":
        return None
    
    # Try to extract from <answer>Option [letter]</answer> format
    match = re.search(r'<answer>Option ([A-E])</answer>', prediction)
    if match:
        return match.group(1)
    
    # Alternative formats
    match = re.search(r'Option ([A-E])', prediction)
    if match:
        return match.group(1)
    
    # Just find any letter
    match = re.search(r'\b([A-E])\b', prediction)
    if match:
        return match.group(1)
    
    return None

# Extract answer letters
results_df['extracted_prediction'] = results_df['GPT4o_prediction'].apply(extract_answer_letter)

# Calculate accuracy
results_df['is_correct'] = results_df['extracted_prediction'] == results_df['answer_corr']

# Overall accuracy
valid_predictions = results_df['extracted_prediction'].notna()
overall_accuracy = results_df.loc[valid_predictions, 'is_correct'].mean() * 100
overall_std = results_df.loc[valid_predictions, 'is_correct'].std() * 100

print(f"\nOverall Accuracy: {overall_accuracy:.2f}%")
print(f"Standard Deviation: {overall_std:.2f}%")
print(f"Total valid predictions: {valid_predictions.sum()} ({valid_predictions.sum()/len(results_df)*100:.2f}%)")

# Analysis by data source
print("\nAccuracy by Data Source:")
data_sources = results_df['data_source_corr'].unique()

for source in data_sources:
    source_df = results_df[results_df['data_source_corr'] == source]
    valid_source_predictions = source_df['extracted_prediction'].notna()
    
    if valid_source_predictions.sum() > 0:
        source_accuracy = source_df.loc[valid_source_predictions, 'is_correct'].mean() * 100
        source_std = source_df.loc[valid_source_predictions, 'is_correct'].std() * 100
        source_count = valid_source_predictions.sum()
        
        print(f"{source}: {source_accuracy:.2f}% (±{source_std:.2f}%), n={source_count}")

# Save the analysis results
results_df.to_csv(paths.REMOVE_LOW_IRR_DATA / "GPT4o_analysis_results.csv", index=False)
print("\nDetailed analysis saved to GPT4o_analysis_results.csv")

Loaded 1300 rows from GPT4o_predictions.csv

Overall Accuracy: 72.30%
Standard Deviation: 44.77%
Total valid predictions: 1148 (88.31%)

Accuracy by Data Source:
jama: 72.51% (±44.69%), n=582
medxpert: 40.96% (±49.33%), n=166
medbullets: 76.81% (±42.31%), n=207
mmlu: 93.78% (±24.21%), n=193

Detailed analysis saved to GPT4o_analysis_results.csv
